# B. Backbone Tuning (No MMDet / No MMCV)

원격 Jupyter 환경(Python 3.12 + 최신 PyTorch)에서는 MMCV wheel이 없어서 MMDet 대신 아래로 진행합니다.

## 이 노트북이 하는 일
1. **torchvision Faster R-CNN** backbone 비교
   - `resnet50_fpn`
   - `resnet50_fpn_v2`
   - `mobilenet_v3_large_fpn`
2. (선택) **Ultralytics YOLO11** 스케일 비교 (`n/s/m`) ≈ capacity/backbone family 튜닝
3. 전처리 파이프라인: `original` / `clahe` / `clahe_gamma`

## Backbone 후보 현실성 (현재 환경 기준)
| 후보 | 이 노트북에서 | 비고 |
|------|----------------|------|
| ResNet50/101 | ✅ | torchvision 공식 |
| MobileNetV3 | ✅ | torchvision 공식 |
| YOLO11 n/s/m/l | ✅ | ultralytics (이미 검증됨) |
| ConvNeXt / Swin | △ | torchvision 분류 모델은 있음, detection head 연결은 커스텀 필요 |
| ViT / InternImage | ❌ 당장 X | MMCV/특수 op 또는 큰 커스텀 필요 |

권장 1차 매트릭스:
- FasterRCNN-ResNet50 + `original`
- FasterRCNN-ResNet50 + `clahe_gamma`
- FasterRCNN-MobileNetV3 + `clahe_gamma`
- (선택) YOLO11s + `clahe_gamma`


In [1]:
# Paths (remote Jupyter)
from pathlib import Path
import json
import random
import time
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    fasterrcnn_resnet50_fpn_v2,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm.auto import tqdm

# cwd = .../ARCADE/notebooks 가정
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
ARCADE_DIR = DATA_DIR / 'arcade' / 'arcade'
PREPROCESS_DIR = DATA_DIR / 'preprocessed'
WORK = PROJECT_DIR / 'tv_backbone_exps'
WORK.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('torch:', torch.__version__)
print('cuda :', torch.version.cuda, '| gpu:', torch.cuda.is_available())
print('PROJECT_DIR:', PROJECT_DIR)
print('WORK       :', WORK)
print('DEVICE     :', DEVICE)


torch: 2.13.0+cu130
cuda : 13.0 | gpu: True
PROJECT_DIR: <PROJECT_ROOT>
WORK       : <PROJECT_ROOT>/tv_backbone_exps
DEVICE     : cuda


In [2]:
# Build stenosis-only index from COCO + image roots

TRAIN_JSON = ARCADE_DIR / 'stenosis' / 'train' / 'annotations' / 'train.json'
VAL_JSON = ARCADE_DIR / 'stenosis' / 'val' / 'annotations' / 'val.json'
RAW_TRAIN = ARCADE_DIR / 'stenosis' / 'train' / 'images'
RAW_VAL = ARCADE_DIR / 'stenosis' / 'val' / 'images'

PIPE_IMG = {
    'original': {'train': RAW_TRAIN, 'val': RAW_VAL},
    'clahe': {
        'train': PREPROCESS_DIR / '01_clahe' / 'train',
        'val': PREPROCESS_DIR / '01_clahe' / 'val',
    },
    'clahe_gamma': {
        'train': PREPROCESS_DIR / '04_clahe_gamma' / 'train',
        'val': PREPROCESS_DIR / '04_clahe_gamma' / 'val',
    },
}

TARGET_COCO_ID = 26  # stenosis


def load_stenosis_records(json_path: Path, image_dir: Path):
    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)
    id2img = {im['id']: im for im in coco['images']}
    anns_by = defaultdict(list)
    for ann in coco['annotations']:
        if ann.get('category_id') != TARGET_COCO_ID:
            continue
        anns_by[ann['image_id']].append(ann)

    records = []
    missing = 0
    for img_id, im in id2img.items():
        path = image_dir / im['file_name']
        if not path.exists():
            missing += 1
            continue
        boxes = []
        for ann in anns_by.get(img_id, []):
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])  # xyxy
        records.append({
            'file_name': im['file_name'],
            'path': str(path),
            'width': im['width'],
            'height': im['height'],
            'boxes': boxes,  # may be empty (negatives)
        })
    return records, missing

# sanity for one pipeline
for pipe, roots in PIPE_IMG.items():
    tr, mtr = load_stenosis_records(TRAIN_JSON, roots['train'])
    va, mva = load_stenosis_records(VAL_JSON, roots['val'])
    ntr_box = sum(len(r['boxes']) for r in tr)
    nva_box = sum(len(r['boxes']) for r in va)
    print(f'{pipe:12s} train={len(tr)} (miss {mtr}) boxes={ntr_box} | val={len(va)} (miss {mva}) boxes={nva_box}')


original     train=1000 (miss 0) boxes=1625 | val=200 (miss 0) boxes=406
clahe        train=1000 (miss 0) boxes=1625 | val=200 (miss 0) boxes=406
clahe_gamma  train=1000 (miss 0) boxes=1625 | val=200 (miss 0) boxes=406


In [3]:
# Dataset / collate for torchvision detection

class StenosisDetDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(r['path']).convert('RGB')
        img = torchvision.transforms.functional.to_tensor(img)  # float [0,1]

        boxes = torch.tensor(r['boxes'], dtype=torch.float32)
        if boxes.ndim == 1:
            boxes = boxes.reshape(0, 4)
        n = boxes.shape[0]
        labels = torch.ones((n,), dtype=torch.int64)  # class 1 = stenosis; 0=bg reserved
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if n else torch.zeros((0,), dtype=torch.float32)
        iscrowd = torch.zeros((n,), dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': area,
            'iscrowd': iscrowd,
        }
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_loaders(pipe: str, batch_size=4, num_workers=2):
    roots = PIPE_IMG[pipe]
    train_rec, _ = load_stenosis_records(TRAIN_JSON, roots['train'])
    val_rec, _ = load_stenosis_records(VAL_JSON, roots['val'])
    train_ds = StenosisDetDataset(train_rec)
    val_ds = StenosisDetDataset(val_rec)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)
    return train_loader, val_loader, train_ds, val_ds

print('dataset helpers ready')


dataset helpers ready


In [4]:
# Model zoo (torchvision Faster R-CNN variants)

BACKBONE_BUILDERS = {
    'resnet50_fpn': lambda: fasterrcnn_resnet50_fpn(weights='DEFAULT'),
    'resnet50_fpn_v2': lambda: fasterrcnn_resnet50_fpn_v2(weights='DEFAULT'),
    'mobilenet_v3_large_fpn': lambda: fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT'),
}


def build_model(backbone_name: str, num_classes=2):
    """num_classes includes background → stenosis => 2"""
    if backbone_name not in BACKBONE_BUILDERS:
        raise KeyError(f'Unknown backbone: {backbone_name}. choices={list(BACKBONE_BUILDERS)}')
    model = BACKBONE_BUILDERS[backbone_name]()
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


print('Available backbones:', list(BACKBONE_BUILDERS))


Available backbones: ['resnet50_fpn', 'resnet50_fpn_v2', 'mobilenet_v3_large_fpn']


In [5]:
# Simple evaluator: VOC-style AP50 on val (no pycocotools dependency required)

def box_iou(a, b):
    # a: [n,4], b:[m,4] xyxy
    if a.numel() == 0 or b.numel() == 0:
        return torch.zeros((a.shape[0], b.shape[0]), device=a.device)
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union.clamp(min=1e-6)


def evaluate_ap50(model, data_loader, device, score_thr=0.05):
    model.eval()
    tps, scores, n_gt = [], [], 0
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for out, tgt in zip(outputs, targets):
                gt = tgt['boxes'].to(device)
                n_gt += gt.shape[0]
                keep = out['scores'] >= score_thr
                pred = out['boxes'][keep]
                sc = out['scores'][keep]
                # sort by score desc
                order = sc.argsort(descending=True)
                pred, sc = pred[order], sc[order]
                matched = set()
                for i in range(pred.shape[0]):
                    scores.append(float(sc[i].item()))
                    if gt.shape[0] == 0:
                        tps.append(0)
                        continue
                    ious = box_iou(pred[i:i+1], gt)[0]
                    j = int(torch.argmax(ious).item())
                    if float(ious[j]) >= 0.5 and j not in matched:
                        tps.append(1)
                        matched.add(j)
                    else:
                        tps.append(0)
    if n_gt == 0 or len(scores) == 0:
        return {'AP50': 0.0, 'n_gt': n_gt, 'n_pred': len(scores)}

    scores = np.array(scores)
    tps = np.array(tps)
    order = np.argsort(-scores)
    tps = tps[order]
    fp = 1 - tps
    tp_cum = np.cumsum(tps)
    fp_cum = np.cumsum(fp)
    recalls = tp_cum / n_gt
    precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
    # VOC 11-point style approx via trapz
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        p = precisions[recalls >= t]
        ap += (p.max() if p.size else 0.0)
    ap /= 11.0
    return {'AP50': float(ap), 'n_gt': int(n_gt), 'n_pred': int(len(scores))}

print('evaluator ready')


evaluator ready


In [8]:
# Train ONE Faster R-CNN experiment

backbone_name = 'resnet50_fpn'   # resnet50_fpn | resnet50_fpn_v2 | mobilenet_v3_large_fpn
pipe = 'clahe_gamma'             # original | clahe | clahe_gamma

EPOCHS = 100                       # smoke: 3~5 / full: 12+
BATCH = 4
LR = 0.005
NUM_WORKERS = 0
PRINT_FREQ = 50

exp_name = f'frcnn_{backbone_name}__{pipe}'
out_dir = WORK / exp_name
out_dir.mkdir(parents=True, exist_ok=True)

train_loader, val_loader, train_ds, val_ds = make_loaders(pipe, batch_size=BATCH, num_workers=NUM_WORKERS)
model = build_model(backbone_name).to(DEVICE)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=0.9, weight_decay=1e-4)
lr_sched = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

history = []
best_ap = -1.0
best_path = out_dir / 'best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'{exp_name} ep{epoch}')
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        # skip fully-empty batch edge cases by ensuring at least tensor shapes ok
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += float(loss.item())
        n_batches += 1
        if n_batches % PRINT_FREQ == 0:
            pbar.set_postfix(loss=loss_sum / n_batches)

    lr_sched.step()
    metrics = evaluate_ap50(model, val_loader, DEVICE)
    row = {
        'epoch': epoch,
        'train_loss': loss_sum / max(n_batches, 1),
        **metrics,
    }
    history.append(row)
    print(row)

    if metrics['AP50'] >= best_ap:
        best_ap = metrics['AP50']
        torch.save({'model': model.state_dict(), 'backbone': backbone_name, 'pipe': pipe, 'epoch': epoch, 'AP50': best_ap}, best_path)

pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
print('best AP50:', best_ap)
print('saved:', best_path)


{'epoch': 1, 'train_loss': 0.3183477641940117, 'AP50': 0.18389888976485597, 'n_gt': 406, 'n_pred': 16964}


{'epoch': 2, 'train_loss': 0.2895199955701828, 'AP50': 0.23667749612008393, 'n_gt': 406, 'n_pred': 8569}


{'epoch': 3, 'train_loss': 0.2678940180540085, 'AP50': 0.26322041212995395, 'n_gt': 406, 'n_pred': 5296}


{'epoch': 4, 'train_loss': 0.22076605007052422, 'AP50': 0.3109045753710932, 'n_gt': 406, 'n_pred': 3277}


{'epoch': 5, 'train_loss': 0.20125644353032113, 'AP50': 0.2967136336255419, 'n_gt': 406, 'n_pred': 3058}


...

{'epoch': 96, 'train_loss': 0.16481532791256906, 'AP50': 0.2846165126854997, 'n_gt': 406, 'n_pred': 2366}


{'epoch': 97, 'train_loss': 0.16528090718388558, 'AP50': 0.2846165126854997, 'n_gt': 406, 'n_pred': 2366}


{'epoch': 98, 'train_loss': 0.16532042661309243, 'AP50': 0.2846165126854997, 'n_gt': 406, 'n_pred': 2366}


{'epoch': 99, 'train_loss': 0.1650118291079998, 'AP50': 0.2846165126854997, 'n_gt': 406, 'n_pred': 2366}


{'epoch': 100, 'train_loss': 0.16487180796265602, 'AP50': 0.2846165126854997, 'n_gt': 406, 'n_pred': 2366}
best AP50: 0.3109045753710932
saved: <PROJECT_ROOT>/tv_backbone_exps/frcnn_resnet50_fpn__clahe_gamma/best.pt


In [7]:
# Optional matrix: several backbone x pipeline combos
MATRIX_NUM_WORKERS = 0
MATRIX_LR = 0.0005
NUM_WORKERS = MATRIX_NUM_WORKERS  # 임시 호환
LR = MATRIX_LR

RUN_MATRIX = True

EXPS = [
    ('resnet50_fpn', 'original'),
    ('resnet50_fpn', 'clahe_gamma'),
    # ('mobilenet_v3_large_fpn', 'clahe_gamma'),
    ('resnet50_fpn_v2', 'clahe_gamma'),
]

MATRIX_EPOCHS = 30
MATRIX_BATCH = 4

def train_one(backbone_name, pipe, epochs=MATRIX_EPOCHS, batch=MATRIX_BATCH):
    exp_name = f'frcnn_{backbone_name}__{pipe}'
    out_dir = WORK / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    train_loader, val_loader, _, _ = make_loaders(pipe, batch_size=batch, num_workers=0)
    model = build_model(backbone_name).to(DEVICE)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=1e-4)
    lr_sched = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(epochs // 2, 1), gamma=0.1)
    history, best_ap = [], -1.0
    best_path = out_dir / 'best.pt'
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, n_batches = 0.0, 0
        for images, targets in tqdm(train_loader, desc=f'{exp_name} ep{epoch}', leave=False):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(v for v in loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += float(loss.item()); n_batches += 1
        lr_sched.step()
        metrics = evaluate_ap50(model, val_loader, DEVICE)
        row = {'epoch': epoch, 'train_loss': loss_sum / max(n_batches, 1), **metrics}
        history.append(row)
        print(exp_name, row)
        if metrics['AP50'] >= best_ap:
            best_ap = metrics['AP50']
            torch.save({'model': model.state_dict(), 'backbone': backbone_name, 'pipe': pipe, 'epoch': epoch, 'AP50': best_ap}, best_path)
    pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
    return {'exp': exp_name, 'best_AP50': best_ap, 'best_ckpt': str(best_path)}

if RUN_MATRIX:
    rows = [train_one(b, p) for b, p in EXPS]
    print(pd.DataFrame(rows))
else:
    print('RUN_MATRIX=False — single-run cell 사용 중')


frcnn_resnet50_fpn__original {'epoch': 1, 'train_loss': 0.3199708413481712, 'AP50': 0.16812234851835522, 'n_gt': 406, 'n_pred': 8900}


frcnn_resnet50_fpn__original {'epoch': 2, 'train_loss': 0.29425969552993775, 'AP50': 0.2090616900977115, 'n_gt': 406, 'n_pred': 7451}


frcnn_resnet50_fpn__original {'epoch': 3, 'train_loss': 0.26666543635725976, 'AP50': 0.2956856003735381, 'n_gt': 406, 'n_pred': 3844}


frcnn_resnet50_fpn__original {'epoch': 4, 'train_loss': 0.24542994275689126, 'AP50': 0.27065107568574637, 'n_gt': 406, 'n_pred': 6279}


frcnn_resnet50_fpn__original {'epoch': 5, 'train_loss': 0.21487492939829828, 'AP50': 0.28330153867546265, 'n_gt': 406, 'n_pred': 2739}


...

frcnn_resnet50_fpn_v2__clahe_gamma {'epoch': 26, 'train_loss': 0.018719765972346068, 'AP50': 0.21520406604554979, 'n_gt': 406, 'n_pred': 345}


frcnn_resnet50_fpn_v2__clahe_gamma {'epoch': 27, 'train_loss': 0.018567059023305774, 'AP50': 0.2173073354849488, 'n_gt': 406, 'n_pred': 371}


frcnn_resnet50_fpn_v2__clahe_gamma {'epoch': 28, 'train_loss': 0.018156955599784853, 'AP50': 0.21237418911837516, 'n_gt': 406, 'n_pred': 359}


frcnn_resnet50_fpn_v2__clahe_gamma {'epoch': 29, 'train_loss': 0.017914278866723178, 'AP50': 0.212820606097917, 'n_gt': 406, 'n_pred': 337}


frcnn_resnet50_fpn_v2__clahe_gamma {'epoch': 30, 'train_loss': 0.01744667062163353, 'AP50': 0.21653669973550657, 'n_gt': 406, 'n_pred': 361}
                                  exp  best_AP50  \
0        frcnn_resnet50_fpn__original   0.295686   
1     frcnn_resnet50_fpn__clahe_gamma   0.290947   
2  frcnn_resnet50_fpn_v2__clahe_gamma   0.268508   

                                           best_ckpt  
0  <PROJECT_ROOT>/tv_backbone_exps/f...  
1  <PROJECT_ROOT>/tv_backbone_exps/f...  
2  <PROJECT_ROOT>/tv_backbone_exps/f...  


In [8]:
# Collect torchvision experiment results
rows = []
for d in sorted(WORK.glob('frcnn_*')):
    if not d.is_dir():
        continue
    hist = d / 'history.csv'
    best = d / 'best.pt'
    ap = None
    if hist.exists():
        df = pd.read_csv(hist)
        if 'AP50' in df.columns and len(df):
            ap = float(df['AP50'].max())
    meta = {}
    if best.exists():
        try:
            ckpt = torch.load(best, map_location='cpu')
            meta = {k: ckpt.get(k) for k in ['backbone', 'pipe', 'epoch', 'AP50']}
            if ap is None:
                ap = meta.get('AP50')
        except Exception as e:
            meta = {'load_error': str(e)}
    rows.append({'exp': d.name, 'best_AP50': ap, **meta, 'dir': str(d)})

df_tv = pd.DataFrame(rows).sort_values('best_AP50', ascending=False) if rows else pd.DataFrame()
print(df_tv.to_string(index=False) if len(df_tv) else 'No frcnn_* results yet. Run train cell.')
df_tv


                                      exp  best_AP50               backbone        pipe  epoch     AP50                                                                                    dir
             frcnn_resnet50_fpn__original   0.295686           resnet50_fpn    original      3 0.295686              <PROJECT_ROOT>/tv_backbone_exps/frcnn_resnet50_fpn__original
          frcnn_resnet50_fpn__clahe_gamma   0.290947           resnet50_fpn clahe_gamma      4 0.290947           <PROJECT_ROOT>/tv_backbone_exps/frcnn_resnet50_fpn__clahe_gamma
       frcnn_resnet50_fpn_v2__clahe_gamma   0.268508        resnet50_fpn_v2 clahe_gamma      3 0.268508        <PROJECT_ROOT>/tv_backbone_exps/frcnn_resnet50_fpn_v2__clahe_gamma
frcnn_mobilenet_v3_large_fpn__clahe_gamma   0.185888 mobilenet_v3_large_fpn clahe_gamma      5 0.185888 <PROJECT_ROOT>/tv_backbone_exps/frcnn_mobilenet_v3_large_fpn__clahe_gamma


,exp,best_AP50,backbone,pipe,epoch,AP50,dir
2,frcnn_resnet50_fpn__original,0.295686,resnet50_fpn,original,3,0.295686,<PROJECT_ROOT>/tv_backbone_exps/f...
1,frcnn_resnet50_fpn__clahe_gamma,0.290947,resnet50_fpn,clahe_gamma,4,0.290947,<PROJECT_ROOT>/tv_backbone_exps/f...
3,frcnn_resnet50_fpn_v2__clahe_gamma,0.268508,resnet50_fpn_v2,clahe_gamma,3,0.268508,<PROJECT_ROOT>/tv_backbone_exps/f...
0,frcnn_mobilenet_v3_large_fpn__clahe_gamma,0.185888,mobilenet_v3_large_fpn,clahe_gamma,5,0.185888,<PROJECT_ROOT>/tv_backbone_exps/f...


## (선택) YOLO11 scale tuning = another backbone/capacity axis

이미 Ultralytics는 이 서버에서 동작 확인됨.
backbone 계열을 바꾸기 어려우면 **YOLO11n/s/m**으로 capacity 튜닝하는 게 가장 빠릅니다.


In [6]:
# Optional YOLO11 scale comparison on one pipeline
RUN_YOLO_SCALES = True  # True로 켜서 실행

YOLO_DIR = DATA_DIR / 'yolo'
RUNS_DIR = PROJECT_DIR / 'runs' / 'detect'
PIPE = 'clahe_gamma'          # yolo folder name
SCALES = ['yolo11n.pt', 'yolo11s.pt']  # add 'yolo11m.pt' if VRAM allows

if RUN_YOLO_SCALES:
    from ultralytics import YOLO
    import yaml
    base = YOLO_DIR / PIPE
    assert (base / 'images' / 'train').exists(), base
    data_yaml = base / 'data.yaml'
    data_yaml.write_text(yaml.safe_dump({
        'path': str(base),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['stenosis'],
    }, sort_keys=False), encoding='utf-8')

    rows = []
    for ckpt in SCALES:
        name = f'{Path(ckpt).stem}__{PIPE}'
        model = YOLO(ckpt)
        res = model.train(
            data=str(data_yaml),
            epochs=50,
            imgsz=640,
            batch=16,
            device=0 if torch.cuda.is_available() else 'cpu',
            project=str(RUNS_DIR),
            name=name,
            exist_ok=True,
            pretrained=True,
            plots=True,
        )
        rows.append({'model': ckpt, 'pipe': PIPE, 'save_dir': str(res.save_dir)})
    print(pd.DataFrame(rows))
else:
    print('RUN_YOLO_SCALES=False')


Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
50 epochs completed in 0.035 hours.
Validating <PROJECT_ROOT>/runs/detect/yolo11n__clahe_gamma/weights/best.pt...
                   all        200        406      0.346      0.404      0.296      0.122
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
Results saved to <PROJECT_ROOT>/runs/detect/yolo11n__clahe_gamma
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
50 epochs completed in 0.044 hours.
Validating <PROJECT_ROOT>/runs/detect/yolo11s__clahe_gamma/weights/best.pt...
                   all        200        406      0.356      0.401       0.31      0.121
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
Results saved to <PROJECT_ROOT>/runs/detect/yolo11s__clahe_gamma


## Next backbone options (without MMDet)

### 지금 바로 가능
1. torchvision Faster R-CNN: ResNet50 / ResNet50-v2 / MobileNetV3  
2. YOLO11 n → s → m (스케일 튜닝)  
3. torchvision RetinaNet / FCOS (같은 데이터로 확장 가능)

### 커스텀하면 가능 (중난이도)
4. `torchvision.models.convnext_tiny` / `swin_t` + 직접 FPN/head 연결  
5. `timm` backbone + simple detection head

### 보류 (환경 이슈)
6. InternImage / 정식 MMDet Swin configs → MMCV 필요

### 추천 진행
1. 이 노트북에서 `resnet50_fpn × original/clahe_gamma`  
2. `mobilenet_v3_large_fpn × clahe_gamma`  
3. 여유 있으면 YOLO11s  
4. 상위 조합 고른 뒤 ConvNeXt/Swin 커스텀은 후순위
